## Concept 3 — Tool Calling Agent (create_agent)
The modern LangChain agent. Simpler than ReAct, more robust, no prompt template needed.

### vs Notebook 1 (LLM + Tools) and Notebook 2 (ReAct)
```
Notebook 1 — LLM + Tools (YOU write the loop):
  response = llm.invoke(query)          ← LLM decides tool
  result   = tool.invoke(args)          ← you execute it
  final    = llm.invoke(msgs + result)  ← you call LLM again
  You write every step.

Notebook 2 — ReAct (AgentExecutor manages loop, but text-format parsing):
  executor.invoke({'input': query})
  Framework manages loop. But: LLM must follow strict Thought/Action format.

Notebook 3 — Tool Calling Agent (create_agent, native tool calling):
  agent.invoke({'messages': [query]})
  Framework manages loop. Uses native LLM tool calling — no text parsing needed.
  Simpler code, more reliable.
```

### create_agent vs create_react_agent
```
create_react_agent  → needs ReAct prompt template, text-format output parsing
create_agent        → no special prompt needed, uses native function calling API
```

### Pipeline
```
Query → Agent (autonomous loop):
  LLM: which tool? → execute tool → observe result → done or more tools?
  → Final Answer
```

### Limitation (what Concept 4 fixes)
Output is a plain string — hard to use programmatically in downstream code.
→ Concept 4 wraps the agent output in a typed Pydantic schema.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain.agents import create_agent

llm = get_llm()
print(f'LLM ready | Tools: {[t.name for t in TOOLS]}')

### Step 1 — Create the Agent
`create_agent` takes the LLM, tools, and a system prompt. It returns a LangGraph agent
that manages the tool-calling loop automatically.

In [ ]:
agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=(
        'You are a helpful assistant. '
        'Always use tools for math, documentation lookups, and weather — never guess.'
    ),
)
print('Agent ready. It autonomously manages its own tool-calling loop.')

### Step 2 — Compare: Manual Loop (Notebook 1) vs Agent (Notebook 3)
Same query, same result — see how much less code the agent needs.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage
from AgentUtils import TOOLS

tool_map = {t.name: t for t in TOOLS}
llm_with_tools = llm.bind_tools(TOOLS)

query = 'What is 144 divided by 12?'

# ---- Manual approach (Notebook 1) ----
messages = [HumanMessage(content=query)]
r = llm_with_tools.invoke(messages)
messages.append(r)
for call in r.tool_calls:
    result = tool_map[call['name']].invoke(call['args'])
    messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
manual_answer = llm_with_tools.invoke(messages).content
print(f'Manual (Notebook 1):  {manual_answer}')

# ---- Agent approach (This Notebook) ----
result = agent.invoke({'messages': [{'role': 'user', 'content': query}]})
agent_answer = result['messages'][-1].content
print(f'Agent  (Notebook 3):  {agent_answer}')

print('\nSame answer — but agent required zero manual tool execution code.')

### Step 3 — Inspect the Agent's Internal Trace
`result['messages']` shows the full internal message sequence the agent produced.

In [ ]:
result = agent.invoke({'messages': [{'role': 'user', 'content': 'What is the weather in Hyderabad?'}]})

print('Agent internal trace:')
for msg in result['messages']:
    cls = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for call in msg.tool_calls:
            print(f'  [{cls}]  → tool call: {call["name"]}({call["args"]})')
    elif hasattr(msg, 'name') and msg.name:
        print(f'  [ToolMessage] ← result:    {msg.content}')
    else:
        short = msg.content[:100] if len(msg.content) > 100 else msg.content
        print(f'  [{cls}]: {short}')

### Step 4 — Multi-Tool Query
The agent calls multiple tools automatically when the query requires them.

In [ ]:
result = agent.invoke({'messages': [{'role': 'user',
    'content': 'Search docs for RAG and also calculate 25 multiplied by 4'}]})

print('Tools called by agent:')
for msg in result['messages']:
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for call in msg.tool_calls:
            print(f'  → {call["name"]}({call["args"]})')

print(f'\nFinal answer: {result["messages"][-1].content}')

### Full Function

In [ ]:
def tool_calling_agent(query: str) -> str:
    result = agent.invoke({'messages': [{'role': 'user', 'content': query}]})
    return result['messages'][-1].content

### Test — Standard Queries

In [ ]:
run_queries(tool_calling_agent, label='Concept 3 — Tool Calling Agent (create_agent)')